# 08. Praca domowa

Czas: ok. 60–90 min, najlepiej w dwóch podejściach (zadania 1–4, potem 5–6 i checklista z zadania 8) + zadanie 7 (bonus), kiedy masz ochotę

Zadania są w kolejności od łatwych. Każde ma pustą komórkę z `# TODO` i podpowiedziami; pisz kod SAM, błędy są OK.
Jeśli utkniesz na 10 minut, zajrzyj do rozwiązań w notebooku 09 (Praca domowa: rozwiązania), porównaj i wróć.

**Czego się nauczysz**
- użyć własnej funkcji (`def`) na całej kolumnie tabeli (`apply`) i policzyć wynik,
- skleić kilka podsumowań w jedną tabelę do raportu (`pd.concat`),
- zliczyć pary wartości (`pd.crosstab`) i zrobić tabelę przestawną (`pivot_table`),
- rozbić kilka kodów odpadów z jednej komórki na osobne wiersze (`explode`),
- sprawdzić własny plik Excel, zanim puścisz na nim notebooki z kursu.

## 0. Wczytanie czystego pliku

Pracujemy na pliku CZYSTYM `data/przetargi_clean.xlsx`, czyli wyniku notebooka 06 (Pandas: czyszczenie danych).
Cenę za Mg liczymy tak samo jak w notebooku 07 (Pandas: analiza): wartość umowy / wolumen, bez przeliczania na rok; cena powyżej 3000 zł to błąd danych i zamieniamy ją na puste pole.

In [ ]:
import pandas as pd   # pandas = biblioteka do tabel; skrót "pd" jak w całym kursie

# wczytujemy czysty plik do tabeli df (df = przyjęta nazwa tabeli w pandas; ścieżka względem folderu kursu, w VS Code otwórz CAŁY folder)
df = pd.read_excel("data/przetargi_clean.xlsx")

# Excel nie pamięta typu Int64, więc liczba ofert wraca jako 1.0, 2.0; astype("Int64") = z powrotem liczba całkowita, która toleruje puste pola
df["liczba_ofert"] = df["liczba_ofert"].astype("Int64")

# cena za Mg = wartość umowy / wolumen; powyżej 3000 zł to błąd danych, więc zamieniamy na puste pole
df["cena_za_mg"] = df["wartosc_pln"] / df["wolumen_mg"]
# df.loc[warunek, "kolumna"] = wartość: wpisuje wartość TYLKO w wierszach, gdzie warunek ma True; None = puste pole
df.loc[df["cena_za_mg"] > 3000, "cena_za_mg"] = None

print("Wiersze:", len(df), "| kolumny:", len(df.columns))
df.head(3)

In [ ]:
# do analiz liczby ofert bierzemy tylko wiersze ze ZNANĄ liczbą ofert; to jedno z założeń przyjętych w notebooku 07 (Pandas: analiza)
# Pułapka: puste pole <NA> wstawione do if daje błąd, dlatego braki usuwamy ZANIM użyjemy własnej funkcji
offers = df.dropna(subset=["liczba_ofert"]).copy()   # dropna(subset=[kolumna]) = wyrzuć wiersze z pustym polem w tej kolumnie; copy() = osobna kopia, bo zaraz dopiszemy nowe kolumny
print("Przetargi ze znaną liczbą ofert:", len(offers), "z", len(df))

## Zadanie 1. Powtórka z Pythona: poziom konkurencji

Funkcja `competition_level` z notebooka 04 (Funkcje), sekcja 3 (Funkcja z if: poziom konkurencji), zamienia liczbę ofert na opis słowny. Użyj jej na całej kolumnie `liczba_ofert` w tabeli `offers`: `apply` wywołuje funkcję dla każdej komórki kolumny i oddaje nową kolumnę wyników. Potem policz, ile przetargów ma każdy poziom.
- Wpisz funkcję jeszcze raz: 1 to "brak konkurencji", 2 to "słaba konkurencja", 3 i więcej to "konkurencja".
- `apply` daje nową kolumnę `poziom_konkurencji`, a `value_counts()` zlicza opisy.

In [ ]:
# Zadanie 1a
# TODO: def competition_level(offers_count): w środku if / elif / else, każda gałąź kończy się return z opisem.
# Podpowiedź: == porównuje, = przypisuje; po każdym warunku dwukropek, w środku 4 spacje wcięcia.
# Sprawdź na pojedynczych liczbach: print(competition_level(1), competition_level(3))

In [ ]:
# Zadanie 1b
# TODO: offers["poziom_konkurencji"] = offers["liczba_ofert"].apply(...)   # w nawiasie sama nazwa funkcji, bez ()
# TODO: offers["poziom_konkurencji"].value_counts()
# Podpowiedź: value_counts(normalize=True) * 100 daje udziały w procentach.

## Zadanie 2. Województwa: trzy liczby w jednej tabeli

Dla każdego województwa policz: liczbę przetargów, medianę ceny za Mg i udział przetargów z jedną ofertą. Zrób trzy osobne `groupby`, a potem sklej wyniki w jedną tabelę: `pd.concat([...], axis=1)`.
- `axis=1` znaczy "sklej OBOK siebie, jako kolumny"; pandas sam dopasuje wiersze po nazwie województwa (jak WYSZUKAJ.PIONOWO w Excelu).
- Udział jednej oferty: porównanie `== 1` na całej kolumnie daje kolumnę `jedna_oferta` z True/False, a `mean()` z niej to udział True, bo True liczy się jak 1, a False jak 0.

In [ ]:
# Zadanie 2a: trzy osobne podsumowania
# TODO: tenders_per_region = df.groupby("wojewodztwo").size()
# TODO: median_price_per_region = df.groupby("wojewodztwo")["..."].median()
# TODO: offers["jedna_oferta"] = offers["liczba_ofert"] == 1
# TODO: single_bid_share_per_region = offers.groupby("...")["jedna_oferta"].mean()
# Podpowiedź: size() liczy wiersze w grupie; median() i mean() pomijają puste pola same.

In [ ]:
# Zadanie 2b: sklejenie w jedną tabelę
# TODO: region_table = pd.concat([...trzy wyniki z 2a...], axis=1)
# TODO: region_table.columns = ["liczba_przetargow", "mediana_ceny_za_mg", "udzial_1_oferty"]
# Podpowiedź: udział pomnóż przez 100 i .round(1), medianę .round(0); na koniec sort_values("udzial_1_oferty", ascending=False), żeby tabela nadawała się do raportu.

## Zadanie 3. Tryb a liczba ofert

`pd.crosstab` zlicza pary wartości: ile przetargów ma dany tryb I daną liczbę ofert (tabela przestawna z samym zliczaniem). W nawiasie podajesz dwie kolumny: pierwsza idzie do wierszy (`tryb`), druga do kolumn (`liczba_ofert`); potem zrób wersję z `normalize="index"` (udziały w wierszu).
- W naszym zbiorze są dwa tryby: `przetarg nieograniczony` i `tryb podstawowy` (rzadszy, w danych dopiero od 2021), więc tabela ma dwa wiersze. Pytanie: czy w trybie podstawowym jedna oferta zdarza się częściej?
- Dla wprawy podstaw potem `wojewodztwo` zamiast `tryb`: kod zostaje ten sam, wierszy będzie 16.

In [ ]:
# Zadanie 3
# TODO: pd.crosstab(offers["..."], offers["..."])   # tryb w wierszach, liczba ofert w kolumnach
# TODO: to samo z normalize="index", pomnożone przez 100 i .round(1)
# Podpowiedź: pierwsza kolumna w nawiasie to wiersze, druga to kolumny; normalize="index" = każdy wiersz sumuje się do 100%.
# TODO (dla wprawy): pd.crosstab(offers["wojewodztwo"], offers["liczba_ofert"])
# Podpowiedź: ten sam kod, tylko 16 wierszy zamiast 2; szukaj województw z największą liczbą w kolumnie 1 (jedna oferta).

## Zadanie 4. Okres umowy a liczba ofert

Czy dłuższe umowy przyciągają więcej ofert? Pogrupuj `offers` po `okres_mies` i policz średnią `liczba_ofert` w każdej grupie (`groupby` = pogrupuj, `mean()` = średnia w każdej grupie).
- Sprawdź też, ile przetargów jest w każdej grupie (`size()`): średnia z kilku przetargów mało znaczy.
- Porównaj wynik z korelacją okresu i liczby ofert z notebooka 07 (Pandas: analiza), sekcja 5 (Korelacje): wyszło tam ok. +0,05, czyli brak związku.

In [ ]:
# Zadanie 4
# TODO: offers.groupby("...")["liczba_ofert"].mean().round(2)
# TODO: offers.groupby("...").size()
# Podpowiedź: oba wyniki możesz skleić tak jak w zadaniu 2 (Województwa: trzy liczby w jednej tabeli): pd.concat([...], axis=1) i nadać nazwy kolumn.

## Zadanie 5. Kody odpadów: najczęstsze kody

W `kody_odpadow` jest kilka kodów w jednej komórce, rozdzielonych średnikiem i spacją. Rozbij je: `str.split(separator)` tnie tekst po separatorze i robi z niego listę (kilka wartości w jednej komórce, w nawiasach kwadratowych), `explode()` rozkłada listę na osobne wiersze, `value_counts()` liczy kody.
- Uwaga: cena za Mg dotyczy CAŁEGO przetargu, nie jednego kodu. Po `explode` nie licz z niej średnich, bo ten sam przetarg policzyłby się kilka razy.

In [ ]:
# Zadanie 5
# TODO: codes_lists = df["..."].str.split("...")   # separator: średnik ze spacją; obejrzyj codes_lists.head(3): w komórkach są listy
# Podpowiedź: w wyniku teksty są w pojedynczych cudzysłowach '20 03 01'; to to samo, co "20 03 01" w kodzie.
# TODO: codes = codes_lists.explode()                        # jeden kod = jeden wiersz
# TODO: codes.value_counts().head(10)
# Podpowiedź: podziel wynik przez len(df) i pomnóż przez 100, a dostaniesz udział przetargów z danym kodem.

## Zadanie 6. Top 5 wykonawców w 2024 wg wartości umów

Wybierz przetargi ogłoszone w 2024 (filtr po `rok`), zsumuj `wartosc_pln` per wykonawca, posortuj malejąco i pokaż 5 pierwszych.
- Podpowiedź: `groupby("wykonawca")["wartosc_pln"].sum()`, potem `sort_values(ascending=False)` i `head(5)`.
- Kwoty pokaż w mln zł (podziel przez `1_000_000` i `round(1)`); bez tego pandas pokaże 591 728 796 zł jako `5.917288e+08` (zapis naukowy: przesuń przecinek o 8 miejsc w prawo).

In [ ]:
# Zadanie 6
# TODO: tenders_2024 = df[df["rok"] == 2024]   # filtr: warunek w nawiasie daje True/False dla każdego wiersza, zostają wiersze z True
# TODO: value_by_contractor = tenders_2024.groupby("...")["..."].sum()
# TODO: top_contractors_2024 = value_by_contractor.sort_values(ascending=False).head(5)
# TODO: (top_contractors_2024 / 1_000_000).round(1)   # kwoty w mln zł
# Podpowiedź: rok to liczba, więc 2024 bez cudzysłowu; ascending=False = od największej.

## Zadanie 7 (bonus). Tabela przestawna: mediana ceny per rok i województwo

`pivot_table` to tabela przestawna z Excela: `index` = wiersze, `columns` = kolumny, `values` = co liczymy, `aggfunc` = jak liczymy.
Zrób: `df.pivot_table(values="cena_za_mg", index="rok", columns="wojewodztwo", aggfunc="median")`. Puste pole (`NaN`) znaczy: w tym roku nie było w tym województwie przetargu ze znaną ceną.

In [ ]:
# Zadanie 7 (bonus)
# TODO: df.pivot_table(values="...", index="...", columns="...", aggfunc="median").round(0)
# Podpowiedź: zamień miejscami index i columns, a tabela będzie węższa (16 wierszy, 6 kolumn) i lepiej wejdzie do raportu.

## Zadanie 8. Checklista: podmieniam na swoje dane

Otwórz notebook 06 (Pandas: czyszczenie danych) obok własnego pliku Excel i wypisz w komentarzach, co musisz zmienić (lista jest w jego sekcji 12 (Jak podmienić na swoje dane)): ścieżka, nazwy kolumn (słownik `COLUMN_NAMES`), format daty, jednostki (Mg czy m³, czyli tony czy metry sześcienne; brutto czy netto; za umowę czy za rok).
Potem wczytaj plik i uruchom trzy kontrole: `info()`, `isna().sum()`, `describe()`. Na trening użyj `data/przetargi_clean.xlsx`.

In [ ]:
# Zadanie 8, kontrola 1
# TODO: wypisz tutaj w komentarzach 4 rzeczy do zmiany w notebooku 06 (Pandas: czyszczenie danych): ścieżka, nazwy kolumn, format daty, jednostki
# TODO: my_df = pd.read_excel("data/przetargi_clean.xlsx")   # u siebie: własna ścieżka; jeśli arkuszy jest kilka, dodaj sheet_name="nazwa arkusza"
# TODO: my_df.info()
# Podpowiedź: w info() typ object (albo str) = tekst; tekst tam, gdzie ma być liczba, to kolumna do czyszczenia.

In [ ]:
# Zadanie 8, kontrola 2
# TODO: my_df.isna().sum()

In [ ]:
# Zadanie 8, kontrola 3
# TODO: my_df.describe(include="number").round(0)   # include="number" = tylko kolumny liczbowe (bez tego pandas dodałby też kolumnę z datą)
# Podpowiedź: patrz na min i max: zero, wartość ujemna albo 10 razy za duża to błąd danych. round(0) = pełne złote zamiast zapisu naukowego typu 1.6e+06.

**Podsumowanie**
- Rozwiązania są w notebooku 09 (Praca domowa: rozwiązania): porównaj ze swoim kodem. Inny zapis z tym samym wynikiem jest w porządku.
- Tabele z zadań 2, 4, 6 i 7 to gotowe fragmenty raportu; do Excela zapisuje je `tabela.to_excel("wyniki/nazwa.xlsx")`.
- Najważniejsze jest zadanie 8: po nim notebooki 06 (Pandas: czyszczenie danych) i 07 (Pandas: analiza) działają na Twoich danych.

Dalej: notebook 09 (Praca domowa: rozwiązania), do porównania po samodzielnej próbie